<a href="https://colab.research.google.com/github/Christy-Roy-tech/-Al-Powered-Robotic-System-for-Early-Plant-Disease-Detection-in-Greenhouses/blob/main/Plant_Disease_Detection_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# ---------------- CONFIG ----------------
DATA_DIR = "/content/plantvillage"   # <-- point this to your filtered dataset folder
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS_HEAD = 8        # stage 1: train classifier head only (frozen backbone)
NUM_EPOCHS_FINETUNE = 5    # stage 2: unfreeze last block, fine-tune at low LR
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
print(f"Using device: {DEVICE}")

# ---------------- DATA ----------------
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

base_dataset = datasets.ImageFolder(DATA_DIR)
class_names = base_dataset.classes
num_classes = len(class_names)
print(f"Found {num_classes} classes:")
for c in class_names:
    print(f"  - {c}")

n_total = len(base_dataset)
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

train_idx, val_idx, test_idx = random_split(
    range(n_total), [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

def make_subset(indices, tfms):
    ds = datasets.ImageFolder(DATA_DIR, transform=tfms)
    return torch.utils.data.Subset(ds, indices.indices)

train_set = make_subset(train_idx, train_tfms)
val_set = make_subset(val_idx, eval_tfms)
test_set = make_subset(test_idx, eval_tfms)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

# ---------------- MODEL ----------------
def build_model(num_classes):
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
    for param in model.features.parameters():
        param.requires_grad = False
    model.classifier[1] = nn.Linear(model.last_channel, num_classes)
    return model

model = build_model(num_classes).to(DEVICE)

# ---------------- TRAIN LOOP ----------------
criterion = nn.CrossEntropyLoss()

def train_model(model, loaders, criterion, optimizer, scheduler, num_epochs):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()
            running_loss, running_corrects, n = 0.0, 0, 0

            for inputs, labels in loaders[phase]:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                n += inputs.size(0)

            if phase == "train" and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / n
            epoch_acc = running_corrects.double() / n
            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc.item())
            print(f"Epoch {epoch+1}/{num_epochs} [{phase}] loss: {epoch_loss:.4f} acc: {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_wts)
    print(f"Best val accuracy: {best_acc:.4f}")
    return model, history

loaders = {"train": train_loader, "val": val_loader}

# Stage 1: train classifier head only
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)
scheduler = lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
model, history1 = train_model(model, loaders, criterion, optimizer, scheduler, NUM_EPOCHS_HEAD)

# Stage 2: unfreeze last few layers, fine-tune at lower LR
for param in list(model.features.parameters())[-20:]:
    param.requires_grad = True

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINETUNE)
scheduler = lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
model, history2 = train_model(model, loaders, criterion, optimizer, scheduler, NUM_EPOCHS_FINETUNE)

# ---------------- EVALUATION ----------------
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("\n=== Classification Report (Test Set) ===")
print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=90)
plt.yticks(range(num_classes), class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Multi-Crop Disease Classifier")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

# training curves
train_acc_all = history1["train_acc"] + history2["train_acc"]
val_acc_all = history1["val_acc"] + history2["val_acc"]
epochs_range = range(1, len(train_acc_all) + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, train_acc_all, label="Train Acc")
plt.plot(epochs_range, val_acc_all, label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Curve")
plt.legend()
plt.savefig("training_curve.png", dpi=150)
plt.show()

# ---------------- SAVE / EXPORT ----------------
torch.save(model.state_dict(), "plant_disease_model.pth")

# Export to ONNX for edge deployment (Raspberry Pi / Jetson compatible runtime)
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
torch.onnx.export(
    model, dummy_input, "plant_disease_model.onnx",
    input_names=["input"], output_names=["output"],
    opset_version=12
)
print("\nSaved: plant_disease_model.pth, plant_disease_model.onnx, "
      "confusion_matrix.png, training_curve.png")
